# Week 6: Two-Tool Calling Pipeline, Direct and over MCP

<a href="https://colab.research.google.com/github/jorshimayor/AI-Engineering-Flow_Research/blob/main/jupyter/week6_tool_calling_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

A tool-calling loop with two tools (`lookup_price`, `calculate`), schema validation and malformed-output repair ([`src/tool_agent.py`](../src/tool_agent.py)), run twice on the same 200 shopping problems: once calling the tools as Python functions, once through an MCP server ([`src/mcp_server.py`](../src/mcp_server.py)). Models: Qwen2.5-1.5B-Instruct and 3B-Instruct. The plan and hypotheses were committed before this run: [`notes/week_6/README.md`](../notes/week_6/README.md).

**Run:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`. About 1 hour. Results save to Google Drive after every batch of 16 problems, so re-running after a disconnect continues where it stopped.

## 1. GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → T4 GPU"
print(torch.cuda.get_device_name(0))

## 2. Code (always the latest)

In [ ]:
REPO = "https://github.com/jorshimayor/AI-Engineering-Flow_Research.git"
BRANCH = "main"
import os
BASE = "/content" if os.path.isdir("/content") else os.getcwd()
CODE = os.path.join(BASE, "AI-Engineering-Flow_Research")
if not os.path.exists(CODE):
    !git clone --depth 1 --branch {BRANCH} {REPO} {CODE}
%cd {CODE}
!git fetch -q --depth 1 origin {BRANCH} && git checkout -q -B {BRANCH} FETCH_HEAD
!pip install -q "transformers==4.49.0" accelerate "mcp==2.3.0" jsonschema
!git log --oneline -1

## 3. Results on Google Drive

In [ ]:
USE_DRIVE = True
RESULTS = "results/week6"
if USE_DRIVE:
    import shutil
    from google.colab import drive
    drive.mount("/content/drive")
    target = "/content/drive/MyDrive/AI-Engineering/week6"
    os.makedirs(target, exist_ok=True)
    if os.path.isdir(RESULTS) and not os.path.islink(RESULTS):   # keep files the repo already has here
        for f in os.listdir(RESULTS):
            if os.path.isfile(os.path.join(RESULTS, f)) and not os.path.exists(os.path.join(target, f)):
                shutil.copy(os.path.join(RESULTS, f), target)
        shutil.rmtree(RESULTS)
    os.makedirs(os.path.dirname(RESULTS), exist_ok=True)
    if not os.path.islink(RESULTS):
        os.symlink(target, RESULTS)
os.makedirs(RESULTS, exist_ok=True)
print("results go to:", os.path.realpath(RESULTS))

## 4. Tools, tasks and the MCP server

In [ ]:
import sys, importlib, json
sys.path.insert(0, "src")
import tools, tool_tasks, tool_agent
for m in (tool_tasks, tools, tool_agent): importlib.reload(m)
assert (tools.TOOLS_VERSION, tool_tasks.TASKS_VERSION) == ("1.0", "1.0"), "stale code: Runtime → Disconnect and delete runtime, then Run all"
PROBLEMS = tool_tasks.make_problems("test", 200)
print(len(PROBLEMS), "test problems, e.g.:", PROBLEMS[0]["question"], "->", PROBLEMS[0]["gold"])
mcp = tool_agent.MCPTools()   # starts src/mcp_server.py and lists its tools
print(json.dumps(mcp.schemas[0], indent=1))
print(mcp.call("lookup_price", {"item": "olive oils"}))
mcp.close()

## 5. Checks before the long run (about 3 minutes)

The model-free tests (parser on known-bad replies, scripted loop over both routes), then 12 **dev** problems on the 1.5B model. Dev problems use catalogue items the test set never uses. This only confirms the setup works: the prompts and hypotheses are already fixed.

In [ ]:
!python tests/test_week6.py
from pathlib import Path
from run_prompt_bench import BatchLLM
from run_tool_bench import run, summary
llm = BatchLLM("Qwen/Qwen2.5-1.5B-Instruct")
_ = summary([run("Qwen/Qwen2.5-1.5B-Instruct", ["direct"], "dev", 12, Path(RESULTS) / "dev" / "Qwen2.5-1.5B-Instruct.jsonl", 12, llm=llm)])

## 6. Run

Both models, both routes, the same 200 problems. Each model is loaded once.

In [ ]:
import gc
MODELS = ["Qwen/Qwen2.5-1.5B-Instruct", "Qwen/Qwen2.5-3B-Instruct"]
for model in MODELS:
    if llm is None or llm.name != model:
        llm = None; gc.collect(); torch.cuda.empty_cache()
        llm = BatchLLM(model)
    run(model, ["direct", "mcp"], "test", 200, Path(RESULTS) / f"{model.split('/')[-1]}.jsonl", 16, llm=llm)

## 7. Results

In [ ]:
_ = summary(sorted(Path(RESULTS).glob("*.jsonl")))

## 8. Figures

In [ ]:
import figures
importlib.reload(figures)
figures.OUT = Path(RESULTS) / "figures"
figures.RES = Path("results")
figures.week6_success_by_route()
figures.week6_malformed_types()
from IPython.display import Image, display
for f in sorted(figures.OUT.glob("week6_*.png")):
    display(Image(str(f)))

## 9. Download

In [ ]:
zip_path = os.path.join(BASE, "week6_results.zip")
!cd results && zip -qr {zip_path} week6/
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Download", zip_path, "from the output panel")

Send the zip back, or unzip it into `results/` in the repo.